# 반도체 일봉 이상징후 탐지: 81개 설정 비교
**목적**: 회귀 학습 구간 3개 × Z 기준 구간 3개 × 가격 임계값 3개 × 거래량 임계값 3개를 비교한다.
결과는 Python 메모리의 DataFrame/dict로 보관한다. DB·CSV 파일을 쓰는 코드는 없다.
커널을 재시작하면 변수와 수집 데이터는 사라진다. 노트북 자체를 저장하면 표시된 출력은 함께 저장될 수 있다.

**처음 실행**: 아래 설치 셀의 주석을 풀어 필요한 패키지를 설치하고 위에서부터 실행한다.
설정 셀의 기업 목록은 교체 가능한 예시이다. 검증 결과를 본 뒤 마지막 테스트 셀에서 설정을 선택한다.

* 가격: 시장별 동종기업 평균에서 자기 기업을 제외 → 시장 영향을 제거 → 기업 기대수익률 회귀.
* 가격 Z: 매일 당시 과거만으로 계산한 예측 잔차를 과거 Z 구간의 평균·표준편차로 표준화.
* 거래량 Z: log1p(거래량)을 과거 Z 구간의 평균·표준편차로 표준화. 거래량은 이동평균 정상모형이다.
* 탐지: abs(price_z) >= 가격 임계값 OR volume_z >= 거래량 임계값.
* 정답 라벨이 없으므로 탐지율·기업별 편중은 운영 지표다. 낮은 탐지율이나 균등한 분포만으로 최적 모델을 선언하지 않는다.
* 모든 Z 평균·표준편차는 오늘을 제외하며, 표준편차는 ddof=1이다.
* stock_return 등 수익률은 소수 단위(0.01=1%), 이름이 _pct로 끝나는 평가 지표는 이미 퍼센트 단위이다.

In [1]:
# 목적: Jupyter 커널에 의존 패키지를 설치한다. 처음 실행할 때 아래 줄의 #를 제거한다.
# 예상 출력: 설치 로그. 이미 설치했다면 실행할 필요가 없다.
%pip install finance-datareader pandas numpy scikit-learn matplotlib ipywidgets

Note: you may need to restart the kernel to use updated packages.


In [2]:
# 목적: 라이브러리와 표시 도구를 준비한다.
# 예상 출력: 실제 실행 환경의 버전. 버전은 실험 설정과 함께 메모리에 기록된다.
import contextlib
import html
import importlib.metadata
import itertools
import platform
import socket
import time
import warnings
from datetime import datetime, timezone

import FinanceDataReader as fdr
import numpy as np
import pandas as pd
import requests
from IPython.display import HTML, display
from sklearn.model_selection import TimeSeriesSplit

ENVIRONMENT = {
    "python": platform.python_version(),
    "finance-datareader": importlib.metadata.version("finance-datareader"),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit-learn": importlib.metadata.version("scikit-learn"),
}
display(pd.Series(ENVIRONMENT, name="version").to_frame())

,version
python,3.11.16
finance-datareader,0.9.202
numpy,2.1.3
pandas,3.0.5
scikit-learn,1.7.2


# 1. 실험 설정과 기업 목록
기본 수집 기간은 2021 ~ 2025년, 최종 테스트는 2025년이다. 2021 ~ 2024년 거래일에
TimeSeriesSplit(n_splits=3, test_size=126)을 적용하여 마지막 3개 검증 구간을 비교한다.
실제 시작·종료일은 수집된 거래일로 계산되어 출력된다. 126일은 약 6개월이며 달력 6개월과 정확히 같지는 않다.

TimeSeriesSplit의 학습 구간은 이용 가능한 과거 전체를 뜻한다. 실제 회귀에는 매일 그중 최근 60/120/180거래일만 쓴다.
검증·테스트에서도 전날까지 새로 관측된 자료로 갱신한다. 하루가 끝난 뒤의 당일 움직임을 평가하는 운영 방식이다.
미래 수익률을 정답으로 쓰는 예측 문제가 아니므로 gap=0을 사용한다.

가장 긴 조합은 회귀 180일 + 과거 예측 잔차 120일이 필요하다. 준비 기간과 전일 가격을 넉넉히 확보한다.
예시 기업은 고정한 비교 집합이다. 과거 시장 전체를 대표하는 편입·편출 이력을 자동 복원하지 않으며 생존편향의 한계가 있다.

In [3]:
# 목적: 후보 81개와 수집/분할 조건을 고정한다.
# 예상 출력: 분석 기업 6개, 비교 기업, 총 실험 개수.
DATA_START = "2021-01-01"
DATA_END = "2025-12-31"
TEST_START = "2025-01-01"

TRAIN_WINDOWS = [60, 120, 180]
Z_WINDOWS = [40, 60, 120]
PRICE_THRESHOLDS = [2.0, 2.5, 3.0]
VOLUME_THRESHOLDS = [2.0, 2.5, 3.0]
N_SPLITS = 3
VALIDATION_DAYS_PER_FOLD = 126
MIN_PEERS = 3
REQUEST_TIMEOUT = 25
REQUEST_RETRIES = 2

# 기본 국내 일봉 제공처. 종목마다 같은 출처와 같은 가격 기준을 사용한다.
PRICE_SOURCE = "NAVER"            # NAVER 또는 YAHOO
PRICE_FIELD = "Close"             # YAHOO 수정종가 사용 시 "Adj Close"
CHECK_CURRENT_LISTING = False      # True이면 FDR 현재 목록으로 시장 분류를 교차 점검
FETCH_MACRO = False                # True이면 마지막 선택 셀에서 환율/WTI를 추가 수집

# is_active=True: 모델 평가 대상. False도 같은 시장의 산업 평균 계산에는 참여한다.
# 아래 6개 분석 기업은 예시이다. 한미반도체(042700)는 KOSPI로 분류한다.
universe = pd.DataFrame([
    ("005930", "삼성전자",       "KOSPI",  True),
    ("000660", "SK하이닉스",     "KOSPI",  True),
    ("000990", "DB하이텍",       "KOSPI",  True),
    ("042700", "한미반도체",     "KOSPI",  True),
    ("240810", "원익IPS",        "KOSDAQ", True),
    ("036930", "주성엔지니어링", "KOSDAQ", True),
    ("003160", "디아이",         "KOSPI",  False),
    ("281820", "케이씨텍",       "KOSPI",  False),
    ("014680", "한솔케미칼",     "KOSPI",  False),
    ("039030", "이오테크닉스",   "KOSDAQ", False),
    ("084370", "유진테크",       "KOSDAQ", False),
    ("095610", "테스",           "KOSDAQ", False),
    ("058470", "리노공업",       "KOSDAQ", False),
    ("005290", "동진쎄미켐",     "KOSDAQ", False),
    ("033640", "네패스",         "KOSDAQ", False),
], columns=["code", "name", "market", "is_active"])
universe["sector"] = "반도체 및 관련 장비·소재(수동 선정)"
universe["universe_version"] = "example_fixed_v1"
MARKET_SYMBOLS = {"KOSPI": "KS11", "KOSDAQ": "KQ11"}
TARGET_CODES = universe.loc[universe["is_active"], "code"].tolist()

assert not universe["code"].duplicated().any()
assert universe["code"].str.fullmatch(r"\d{6}").all()
assert set(universe["market"]).issubset(MARKET_SYMBOLS)
assert len(TARGET_CODES) > 0
assert pd.Timestamp(DATA_START) < pd.Timestamp(TEST_START) <= pd.Timestamp(DATA_END)
assert PRICE_SOURCE in {"NAVER", "YAHOO"}
assert PRICE_FIELD in {"Close", "Adj Close"}
if PRICE_SOURCE == "NAVER" and PRICE_FIELD != "Close":
    raise ValueError("NAVER에는 별도 Adj Close 컬럼이 없습니다.")
for market, group in universe.groupby("market"):
    assert len(group) - 1 >= MIN_PEERS, f"{market}: 비교 기업을 더 넣어 주세요."

experiment_grid = pd.DataFrame(
    itertools.product(TRAIN_WINDOWS, Z_WINDOWS, PRICE_THRESHOLDS, VOLUME_THRESHOLDS),
    columns=["train_window", "z_window", "price_threshold", "volume_threshold"],
)
experiment_grid.insert(0, "experiment_id", [
    f"R{int(r)}_Z{int(z)}_P{p:.1f}_V{v:.1f}"
    for r, z, p, v in experiment_grid.itertuples(index=False, name=None)
])
print(f"총 {len(experiment_grid)}개 실험 = 학습 구간 3 × Z 구간 3 × 임계값 조합 9")
display(universe)

총 81개 실험 = 학습 구간 3 × Z 구간 3 × 임계값 조합 9


,code,name,market,is_active,sector,universe_version
0,005930,삼성전자,KOSPI,True,반도체 및 관련 장비·소재(수동 선정),example_fixed_v1
1,000660,SK하이닉스,KOSPI,True,반도체 및 관련 장비·소재(수동 선정),example_fixed_v1
2,000990,DB하이텍,KOSPI,True,반도체 및 관련 장비·소재(수동 선정),example_fixed_v1
3,042700,한미반도체,KOSPI,True,반도체 및 관련 장비·소재(수동 선정),example_fixed_v1
4,240810,원익IPS,KOSDAQ,True,반도체 및 관련 장비·소재(수동 선정),example_fixed_v1
5,036930,주성엔지니어링,KOSDAQ,True,반도체 및 관련 장비·소재(수동 선정),example_fixed_v1
6,003160,디아이,KOSPI,False,반도체 및 관련 장비·소재(수동 선정),example_fixed_v1
7,281820,케이씨텍,KOSPI,False,반도체 및 관련 장비·소재(수동 선정),example_fixed_v1
8,014680,한솔케미칼,KOSPI,False,반도체 및 관련 장비·소재(수동 선정),example_fixed_v1
9,039030,이오테크닉스,KOSDAQ,False,반도체 및 관련 장비·소재(수동 선정),example_fixed_v1


# 2. FDR 수집과 품질 점검
수집은 한 번 하고 81개 실험이 같은 원본을 재사용한다. 실패한 기업을 몰래 제외하지 않는다.
오류가 발생하면 메시지에 있는 종목/지수부터 확인하고 재실행한다. 커널이 살아 있으면 성공한 요청은 메모리 캐시를 재사용한다.

기본 NAVER:코드의 Close는 네이버 제공 가격이다. 별도 조정계수/원종가를 동시에 제공한다고 가정하지 않는다.
분할·병합일 주변의 가격과 거래량 기준을 확인한다. 거래량은 분할 전후 주식 수 변화만으로도 이상으로 잡힐 수 있다.
YAHOO와 Adj Close를 선택하면 Yahoo 수정종가 수익률을 사용하며 원시 종가는 close에 따로 남긴다.
모든 기업에 같은 출처를 적용하고 서로 다른 출처로 자동 대체하지 않는다.

In [4]:
# 목적: 타임아웃/재시도/메모리 캐시를 갖춘 수집 함수를 정의한다.
# 예상 출력: 함수 정의만 수행. 데이터나 파일은 아직 생성하지 않는다.
if "download_cache" not in globals():
    download_cache = {}


@contextlib.contextmanager
def network_deadline(seconds):
    """FDR의 requests와 pandas URL 조회에 기본 타임아웃을 한시적으로 적용한다."""
    original_request = requests.sessions.Session.request
    original_socket_timeout = socket.getdefaulttimeout()

    def with_timeout(self, method, url, **kwargs):
        if kwargs.get("timeout") is None:
            kwargs["timeout"] = seconds
        return original_request(self, method, url, **kwargs)

    requests.sessions.Session.request = with_timeout
    socket.setdefaulttimeout(seconds)
    try:
        yield
    finally:
        requests.sessions.Session.request = original_request
        socket.setdefaulttimeout(original_socket_timeout)


def read_fdr(symbol, start, end):
    key = (symbol, str(start), str(end))
    if key in download_cache:
        return download_cache[key].copy()
    error = None
    for attempt in range(REQUEST_RETRIES):
        try:
            # 출처별 종료일 포함 여부 차이를 피하기 위해 하루 더 요청한 후 직접 자른다.
            request_end = (pd.Timestamp(end) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
            with network_deadline(REQUEST_TIMEOUT):
                frame = fdr.DataReader(symbol, start, request_end)
            if frame is None or frame.empty:
                raise ValueError("빈 데이터 반환")
            frame = frame.copy()
            frame.index = pd.to_datetime(frame.index)
            if frame.index.tz is not None:
                frame.index = frame.index.tz_localize(None)
            frame.index = frame.index.normalize()
            if frame.index.duplicated().any():
                raise ValueError("날짜 중복: 원본 확인 필요")
            frame = frame.sort_index().loc[pd.Timestamp(start):pd.Timestamp(end)]
            if frame.empty:
                raise ValueError("요청 기간에 데이터 없음")
            frame.index.name = "date"
            download_cache[key] = frame.copy()
            return frame
        except Exception as exc:
            error = exc
            if attempt + 1 < REQUEST_RETRIES:
                time.sleep(0.5 * (attempt + 1))
    raise RuntimeError(f"FDR 조회 실패 [{symbol}] {type(error).__name__}: {error}") from error


def stock_symbol(code, market):
    if PRICE_SOURCE == "NAVER":
        return f"NAVER:{code}"
    suffix = ".KS" if market == "KOSPI" else ".KQ"
    return f"YAHOO:{code}{suffix}"


def collect_prices(universe_frame):
    stock_frames, index_frames, logs = {}, {}, []
    for row in universe_frame.itertuples(index=False):
        symbol = stock_symbol(row.code, row.market)
        try:
            data = read_fdr(symbol, DATA_START, DATA_END)
            required = {"Close", "Volume", PRICE_FIELD}
            if not required.issubset(data.columns):
                raise ValueError(f"필수 컬럼 누락: {required - set(data.columns)}")
            stock_frames[row.code] = data
            logs.append({"key": row.code, "symbol": symbol, "status": "ok", "rows": len(data), "error": ""})
        except Exception as exc:
            logs.append({"key": row.code, "symbol": symbol, "status": "error", "rows": 0, "error": str(exc)})
        print(f"수집 {row.code} {row.name}: {logs[-1]['status']}")
    for market, symbol in MARKET_SYMBOLS.items():
        try:
            data = read_fdr(symbol, DATA_START, DATA_END)
            if "Close" not in data:
                raise ValueError("지수 Close 누락")
            index_frames[market] = data
            logs.append({"key": market, "symbol": symbol, "status": "ok", "rows": len(data), "error": ""})
        except Exception as exc:
            logs.append({"key": market, "symbol": symbol, "status": "error", "rows": 0, "error": str(exc)})
    return stock_frames, index_frames, pd.DataFrame(logs)

In [5]:
# 목적: 분석·비교 기업 및 두 시장지수를 수집한다.
# 예상 출력: 수집 로그와 오류 목록. 필수 수집에 실패하면 다음 실험으로 진행하지 않는다.
if CHECK_CURRENT_LISTING:
    with network_deadline(REQUEST_TIMEOUT):
        current_listing = fdr.StockListing("KRX")
    current_listing["Code"] = current_listing["Code"].astype(str).str.zfill(6)
    check = universe.merge(current_listing[["Code", "Market"]], left_on="code", right_on="Code", how="left")
    mismatch = check[check["market"] != check["Market"]]
    display(check[["code", "name", "market", "Market"]])
    if not mismatch.empty:
        raise ValueError("현재 목록과 시장 분류가 다르거나 누락되었습니다. 기업 목록을 확인하세요.")

raw_stock_frames, raw_index_frames, download_log = collect_prices(universe)
display(download_log)
if download_log["status"].ne("ok").any():
    raise RuntimeError("수집 오류를 해결한 후 이 셀을 다시 실행하세요. 실패한 기업은 자동 제외하지 않습니다.")

수집 005930 삼성전자: ok
수집 000660 SK하이닉스: ok
수집 000990 DB하이텍: ok
수집 042700 한미반도체: ok
수집 240810 원익IPS: ok
수집 036930 주성엔지니어링: ok
수집 003160 디아이: ok
수집 281820 케이씨텍: ok
수집 014680 한솔케미칼: ok
수집 039030 이오테크닉스: ok
수집 084370 유진테크: ok
수집 095610 테스: ok
수집 058470 리노공업: ok
수집 005290 동진쎄미켐: ok
수집 033640 네패스: ok


,key,symbol,status,rows,error
0,005930,NAVER:005930,ok,1225,
1,000660,NAVER:000660,ok,1225,
2,000990,NAVER:000990,ok,1225,
3,042700,NAVER:042700,ok,1225,
4,240810,NAVER:240810,ok,1225,
5,036930,NAVER:036930,ok,1225,
6,003160,NAVER:003160,ok,1225,
7,281820,NAVER:281820,ok,1225,
8,014680,NAVER:014680,ok,1225,
9,039030,NAVER:039030,ok,1225,


In [6]:
# 목적: 공통 거래일에 맞추고 원본/분석 가격, 수익률, 거래량을 분리한다.
# 예상 출력: 함수 정의. 결측을 앞 값으로 메우지 않으며 0거래량일은 분석에서 제외한다.
def prepare_panels(stock_frames, index_frames, universe_frame):
    calendar = pd.DatetimeIndex(sorted(set().union(*(set(x.index) for x in index_frames.values()))), name="date")
    market_parts, stock_parts, quality = [], [], []
    for market, raw in index_frames.items():
        close = pd.to_numeric(raw["Close"], errors="coerce").reindex(calendar)
        close = close.where(np.isfinite(close) & close.gt(0))
        market_parts.append(pd.DataFrame({
            "date": calendar, "market": market, "index_code": MARKET_SYMBOLS[market],
            "close_value": close.values,
            "market_return": close.pct_change(fill_method=None).values,
        }))
    for row in universe_frame.itertuples(index=False):
        raw = stock_frames[row.code]
        close = pd.to_numeric(raw["Close"], errors="coerce").reindex(calendar)
        price = pd.to_numeric(raw[PRICE_FIELD], errors="coerce").reindex(calendar)
        volume = pd.to_numeric(raw["Volume"], errors="coerce").reindex(calendar)
        usable = np.isfinite(price) & price.gt(0) & np.isfinite(volume) & volume.gt(0)
        model_price = price.where(usable)
        returns = model_price.pct_change(fill_method=None)
        part = pd.DataFrame({
            "date": calendar, "code": row.code, "name": row.name, "market": row.market,
            "is_active": row.is_active, "close": close.values, "model_price": model_price.values,
            "volume": volume.values, "stock_return": returns.values,
            "log_volume": np.log1p(volume.where(usable)).values,
            "usable_quote": usable.values,
            "price_source": PRICE_SOURCE, "price_field": PRICE_FIELD,
        })
        if "Adj Close" in raw:
            part["adjusted_close"] = pd.to_numeric(raw["Adj Close"], errors="coerce").reindex(calendar).values
        else:
            part["adjusted_close"] = np.nan
        stock_parts.append(part)
        quality.append({
            "code": row.code, "name": row.name, "market": row.market,
            "first_date": raw.index.min(), "last_date": raw.index.max(), "raw_rows": len(raw),
            "calendar_rows": len(calendar), "unusable_quotes": int((~usable).sum()),
            "zero_volume_days": int(volume.eq(0).sum()),
            "missing_returns": int(returns.isna().sum()),
            "large_return_review_days": int(returns.abs().gt(0.35).sum()),
        })
    return calendar, pd.concat(stock_parts, ignore_index=True), pd.concat(market_parts, ignore_index=True), pd.DataFrame(quality)


calendar, stock_daily, market_daily, quality_report = prepare_panels(raw_stock_frames, raw_index_frames, universe)
price_review = stock_daily.loc[stock_daily["stock_return"].abs().gt(0.35),
                               ["date", "code", "name", "close", "model_price", "stock_return", "volume"]].copy()
display(quality_report)
print("절대 수익률 35% 초과: 기업행위/데이터 기준을 검토할 날짜. 자동 삭제하지 않습니다.")
display(price_review)
display(market_daily.groupby("market").agg(first_date=("date", "min"), last_date=("date", "max"),
                                          rows=("date", "size"), missing_returns=("market_return", lambda x: x.isna().sum())))

,code,name,market,first_date,last_date,raw_rows,calendar_rows,unusable_quotes,zero_volume_days,missing_returns,large_return_review_days
0,005930,삼성전자,KOSPI,2021-01-04,2025-12-30,1225,1225,0,0,1,0
1,000660,SK하이닉스,KOSPI,2021-01-04,2025-12-30,1225,1225,0,0,1,0
2,000990,DB하이텍,KOSPI,2021-01-04,2025-12-30,1225,1225,0,0,1,0
3,042700,한미반도체,KOSPI,2021-01-04,2025-12-30,1225,1225,3,3,5,0
4,240810,원익IPS,KOSDAQ,2021-01-04,2025-12-30,1225,1225,0,0,1,0
5,036930,주성엔지니어링,KOSDAQ,2021-01-04,2025-12-30,1225,1225,0,0,1,0
6,003160,디아이,KOSPI,2021-01-04,2025-12-30,1225,1225,0,0,1,0
7,281820,케이씨텍,KOSPI,2021-01-04,2025-12-30,1225,1225,0,0,1,0
8,014680,한솔케미칼,KOSPI,2021-01-04,2025-12-30,1225,1225,0,0,1,0
9,039030,이오테크닉스,KOSDAQ,2021-01-04,2025-12-30,1225,1225,0,0,1,0


절대 수익률 35% 초과: 기업행위/데이터 기준을 검토할 날짜. 자동 삭제하지 않습니다.


,date,code,name,close,model_price,stock_return,volume


,first_date,last_date,rows,missing_returns
market,,,,
KOSDAQ,2021-01-04,2025-12-30,1225,1
KOSPI,2021-01-04,2025-12-30,1225,1


# 3. 시장별 산업 벤치마크
각 분석 기업과 같은 시장의 비교 기업을 동일가중한다. 자기 기업을 제외한다.
당일 수익률을 계산할 수 있는 비교 기업이 MIN_PEERS개 미만이면 산업 수익률을 결측으로 둔다.
유효 기업 수와 전체 비교 집합을 보관하여 평균의 구성 변화도 점검할 수 있다.
기본 예시의 한솔케미칼 등은 복수 사업을 보유한다. 팀의 반도체 범위에 맞춰 포함 여부를 결정한다.

In [7]:
# 목적: 기업별 leave-one-out 산업 평균과 회귀 입력 표를 만든다.
# 예상 출력: 비교 기업 목록, 시장별 산업 수익률과 당일 유효 비교 기업 수.
def make_benchmarks(stock_frame, market_frame, universe_frame, min_peers):
    wide = stock_frame.pivot(index="date", columns="code", values="stock_return").sort_index()
    parts, members = [], []
    for target in universe_frame.loc[universe_frame["is_active"]].itertuples(index=False):
        peers = universe_frame.loc[(universe_frame["market"] == target.market) &
                                   (universe_frame["code"] != target.code), "code"].tolist()
        peer_returns = wide[peers]
        counts = peer_returns.notna().sum(axis=1)
        sector = peer_returns.mean(axis=1).where(counts >= min_peers)
        parts.append(pd.DataFrame({"date": wide.index, "code": target.code,
                                   "sector_return": sector.values, "peer_count": counts.values,
                                   "peer_pool_size": len(peers)}))
        members.extend({"target_code": target.code, "peer_code": code, "market": target.market} for code in peers)
    bench = pd.concat(parts, ignore_index=True)
    panel = stock_frame.loc[stock_frame["is_active"]].merge(bench, on=["date", "code"], validate="one_to_one")
    panel = panel.merge(market_frame[["date", "market", "index_code", "market_return"]],
                        on=["date", "market"], validate="many_to_one")
    return panel.sort_values(["code", "date"]).reset_index(drop=True), bench, pd.DataFrame(members)


model_panel, benchmark_daily, benchmark_members = make_benchmarks(stock_daily, market_daily, universe, MIN_PEERS)
display(benchmark_members.groupby(["target_code", "market"])["peer_code"].agg(list).to_frame())
display(benchmark_daily.groupby("code").agg(min_peer_count=("peer_count", "min"),
                                          missing_sector_days=("sector_return", lambda x: x.isna().sum())))

,,peer_code
target_code,market,
000660,KOSPI,"[005930, 000990, 042700, 003160, 281820, 014680]"
000990,KOSPI,"[005930, 000660, 042700, 003160, 281820, 014680]"
005930,KOSPI,"[000660, 000990, 042700, 003160, 281820, 014680]"
036930,KOSDAQ,"[240810, 039030, 084370, 095610, 058470, 00529..."
042700,KOSPI,"[005930, 000660, 000990, 003160, 281820, 014680]"
240810,KOSDAQ,"[036930, 039030, 084370, 095610, 058470, 00529..."


,min_peer_count,missing_sector_days
code,,
000660,0,1
000990,0,1
005930,0,1
036930,0,1
042700,0,1
240810,0,1


In [8]:
# 목적: 날짜 단위 TimeSeriesSplit을 만들고 최종 테스트를 별도로 떼어 둔다.
# 예상 출력: 각 fold의 학습/검증 시작·종료일, 최종 테스트 시작·종료일.
# 기업별 행을 바로 TimeSeriesSplit에 넣지 않는다. 같은 날짜의 모든 기업은 같은 fold에 들어간다.
def make_date_splits(trading_dates, test_start, n_splits, validation_days, max_warmup):
    dev_dates = trading_dates[trading_dates < pd.Timestamp(test_start)]
    heldout_dates = trading_dates[trading_dates >= pd.Timestamp(test_start)]
    if len(heldout_dates) == 0:
        raise ValueError("최종 테스트 날짜가 없습니다. TEST_START/DATA_END를 확인하세요.")
    splitter = TimeSeriesSplit(n_splits=n_splits, test_size=validation_days, gap=0)
    rows, date_rows = [], []
    for fold, (train_idx, valid_idx) in enumerate(splitter.split(dev_dates), start=1):
        train_dates, valid_dates = dev_dates[train_idx], dev_dates[valid_idx]
        if len(train_dates) <= max_warmup:
            raise ValueError("첫 학습 구간이 준비 기간보다 짧습니다. DATA_START를 앞당기거나 검증 길이를 줄이세요.")
        assert train_dates.max() < valid_dates.min() < pd.Timestamp(test_start)
        rows.append({"fold": fold, "history_start": train_dates.min(), "history_end": train_dates.max(),
                     "history_days": len(train_dates), "validation_start": valid_dates.min(),
                     "validation_end": valid_dates.max(), "validation_days": len(valid_dates)})
        date_rows.extend({"date": d, "fold": fold, "split": "validation"} for d in valid_dates)
    assignments = pd.DataFrame(date_rows)
    assert not assignments["date"].duplicated().any()
    return dev_dates, heldout_dates, pd.DataFrame(rows), assignments


development_dates, test_dates, split_plan, validation_date_map = make_date_splits(
    calendar, TEST_START, N_SPLITS, VALIDATION_DAYS_PER_FOLD, max(TRAIN_WINDOWS) + max(Z_WINDOWS) + 1)
display(split_plan)
display(pd.DataFrame([{"split": "FINAL_TEST", "start": test_dates.min(), "end": test_dates.max(), "days": len(test_dates)}]))
print("검증/테스트의 하루가 끝나면 그날 관측치는 다음 날 학습에 사용할 수 있습니다.")

,fold,history_start,history_end,history_days,validation_start,validation_end,validation_days
0,1,2021-01-04,2023-06-14,605,2023-06-15,2023-12-15,126
1,2,2021-01-04,2023-12-15,731,2023-12-18,2024-06-25,126
2,3,2021-01-04,2024-06-25,857,2024-06-26,2024-12-30,126


,split,start,end,days
0,FINAL_TEST,2025-01-02,2025-12-30,242


검증/테스트의 하루가 끝나면 그날 관측치는 다음 날 학습에 사용할 수 있습니다.


# 4. 과거만 사용하는 롤링 회귀와 Z점수
날짜 t에 과거 L거래일로 두 회귀를 추정한다.
1. sector = sector_alpha + sector_beta × market + sector_pure
2. stock_return = alpha + beta_market × market + beta_sector × sector_pure + residual

산업 회귀는 해당 L일 안에서 학습하고 같은 계수로 학습 구간과 오늘의 순산업 수익률을 계산한다.
오늘 시장/산업 수익률은 장 마감 후 관측한 조건으로 사용하며, 회귀계수 추정에는 오늘 기업 수익률이 들어가지 않는다.
가격 Z 기준은 과거 시점에 각각 계산해 둔 순차 예측 잔차다. 전체 기간에 한 번 회귀한 잔차를 재사용하지 않는다.
기본 구현은 정확히 직전 L거래일이 모두 유효할 때만 회귀한다. 결측 행을 지워 시간 간격을 압축하지 않는다.
결측 때문에 평가에서 빠지는 기업·날짜는 뒤의 coverage 표에 표시한다.

In [9]:
# 목적: 회귀와 Z 계산 함수를 정의한다.
# 예상 출력: 함수 정의. 실제 계산은 다음 셀에서 3개 회귀 구간 × 3개 Z 구간에 대해 수행한다.
def fit_rolling_one_company(company_panel, train_window):
    frame = company_panel.sort_values("date").reset_index(drop=True).copy()
    n = len(frame)
    output_cols = ["sector_alpha", "sector_beta", "sector_pure_return", "alpha", "beta_market",
                   "beta_sector", "expected_return", "price_residual", "regression_train_n"]
    for col in output_cols:
        frame[col] = np.nan
    frame["regression_train_start"] = pd.NaT
    frame["regression_train_end"] = pd.NaT
    frame["regression_status"] = "warmup"
    values = frame[["stock_return", "market_return", "sector_return"]].to_numpy(float)
    for t in range(train_window, n):
        train = values[t-train_window:t]
        today = values[t]
        if not np.isfinite(train).all() or not np.isfinite(today).all():
            frame.at[t, "regression_status"] = "missing_data"
            continue
        market_train, sector_train, stock_train = train[:, 1], train[:, 2], train[:, 0]
        market_design = np.column_stack([np.ones(train_window), market_train])
        sector_coef, _, rank1, _ = np.linalg.lstsq(market_design, sector_train, rcond=None)
        sector_pure_train = sector_train - market_design @ sector_coef
        design = np.column_stack([np.ones(train_window), market_train, sector_pure_train])
        coef, _, rank2, _ = np.linalg.lstsq(design, stock_train, rcond=None)
        if rank1 != 2 or rank2 != 3:
            frame.at[t, "regression_status"] = "rank_deficient"
            continue
        pure_today = today[2] - (sector_coef[0] + sector_coef[1] * today[1])
        expected = coef[0] + coef[1] * today[1] + coef[2] * pure_today
        numbers = [sector_coef[0], sector_coef[1], pure_today, coef[0], coef[1], coef[2],
                   expected, today[0] - expected, train_window]
        frame.loc[t, output_cols] = numbers
        frame.at[t, "regression_train_start"] = frame.at[t-train_window, "date"]
        frame.at[t, "regression_train_end"] = frame.at[t-1, "date"]
        frame.at[t, "regression_status"] = "ok"
    frame["train_window"] = train_window
    return frame


def add_past_zscores(regression_frame, z_window):
    parts = []
    for _, company in regression_frame.groupby("code", sort=False):
        company = company.sort_values("date").reset_index(drop=True).copy()
        past_residual = company["price_residual"].shift(1).rolling(z_window, min_periods=z_window)
        company["price_residual_mean"] = past_residual.mean()
        company["price_residual_std"] = past_residual.std(ddof=1)
        price_std = company["price_residual_std"].where(company["price_residual_std"] > 1e-12)
        company["price_z"] = (company["price_residual"] - company["price_residual_mean"]) / price_std
        past_log_volume = company["log_volume"].shift(1).rolling(z_window, min_periods=z_window)
        company["expected_log_volume"] = past_log_volume.mean()
        company["log_volume_std"] = past_log_volume.std(ddof=1)
        company["volume_residual"] = company["log_volume"] - company["expected_log_volume"]
        volume_std = company["log_volume_std"].where(company["log_volume_std"] > 1e-12)
        company["volume_z"] = company["volume_residual"] / volume_std
        valid_volume = company["volume"].where(company["usable_quote"])
        company["past_mean_volume"] = valid_volume.shift(1).rolling(z_window, min_periods=z_window).mean()
        company["volume_ratio"] = company["volume"] / company["past_mean_volume"]
        company["z_reference_start"] = company["date"].shift(z_window)
        company["z_reference_end"] = company["date"].shift(1)
        company["z_window"] = z_window
        company["model_valid"] = np.isfinite(company[["price_z", "volume_z"]]).all(axis=1)
        parts.append(company)
    return pd.concat(parts, ignore_index=True)


def build_feature_cache(panel, train_windows, z_windows):
    cache = {}
    for train_window in train_windows:
        regression = pd.concat([fit_rolling_one_company(group, train_window)
                                for _, group in panel.groupby("code", sort=False)], ignore_index=True)
        for z_window in z_windows:
            cache[(train_window, z_window)] = add_past_zscores(regression, z_window)
        print(f"회귀 {train_window}일 / Z {z_windows}일 계산 완료")
    return cache

In [10]:
# 목적: 최종 테스트 이전 데이터만으로 검증용 점수를 미리 계산한다.
# 예상 출력: 9개 구간 조합의 점수 표가 feature_cache에 저장된다. 임계값은 이후 이 점수에 적용한다.
development_panel = model_panel.loc[model_panel["date"] < pd.Timestamp(TEST_START)].copy()
feature_cache = build_feature_cache(development_panel, TRAIN_WINDOWS, Z_WINDOWS)
print("feature_cache 키:", list(feature_cache))

회귀 60일 / Z [40, 60, 120]일 계산 완료
회귀 120일 / Z [40, 60, 120]일 계산 완료
회귀 180일 / Z [40, 60, 120]일 계산 완료
feature_cache 키: [(60, 40), (60, 60), (60, 120), (120, 40), (120, 60), (120, 120), (180, 40), (180, 60), (180, 120)]


In [11]:
# 목적: 9개 구간 설정 모두 점수를 낼 수 있는 공통 기업·날짜를 정한다.
# 예상 출력: 설정별 계산 가능 수와 공통 평가 표본 수. 결측/준비 기간을 정상으로 세지 않는다.
def common_eligibility(cache):
    valid_columns = []
    for (train_window, z_window), frame in cache.items():
        valid = frame.set_index(["date", "code"])["model_valid"].rename(f"R{train_window}_Z{z_window}")
        assert valid.index.is_unique
        valid_columns.append(valid)
    validity = pd.concat(valid_columns, axis=1).fillna(False).astype(bool)
    common = validity.all(axis=1).rename("common_valid").reset_index()
    return common, validity


common_valid, validity_matrix = common_eligibility(feature_cache)
validation_pairs = model_panel[["date", "code", "name"]].merge(validation_date_map, on="date", validate="many_to_one")
coverage = validation_pairs.merge(common_valid, on=["date", "code"], how="left", validate="one_to_one")
coverage["common_valid"] = coverage["common_valid"].fillna(False).astype(bool)
coverage_report = coverage.groupby(["code", "name"], as_index=False).agg(
    requested_rows=("date", "size"), evaluated_rows=("common_valid", "sum"))
coverage_report["excluded_rows"] = coverage_report["requested_rows"] - coverage_report["evaluated_rows"]
coverage_report["coverage_pct"] = 100 * coverage_report["evaluated_rows"] / coverage_report["requested_rows"]
display(coverage_report)
if coverage_report["evaluated_rows"].eq(0).any():
    raise ValueError("검증 가능한 날짜가 없는 분석 기업이 있습니다. 결측·준비 기간·기업 구성을 점검하세요.")
for fold in validation_date_map["fold"].unique():
    if not coverage.loc[coverage["fold"].eq(fold), "common_valid"].any():
        raise ValueError(f"fold {fold}: 공통 유효 표본이 없습니다.")

,code,name,requested_rows,evaluated_rows,excluded_rows,coverage_pct
0,000660,SK하이닉스,378,378,0,100.000000
1,000990,DB하이텍,378,378,0,100.000000
2,005930,삼성전자,378,378,0,100.000000
3,036930,주성엔지니어링,378,378,0,100.000000
4,042700,한미반도체,378,372,6,98.412698
5,240810,원익IPS,378,378,0,100.000000


# 5. 81개 실험과 평가 지표
**탐지율 분모**: 모든 설정에서 공통으로 평가할 수 있는 (기업, 거래일) 개수.
거래일 100개 × 기업 6개가 모두 유효하면 분모는 600이다. 가격·거래량이 동시에 이상이어도 OR 탐지 건수는 1건이다.

**기업별 편중**: ① 각 기업의 탐지율 ② 전체 경보에서 차지하는 비중 ③ 표본 비중 대비 경보 비중(lift)을 같이 본다.
기업별 탐지율 최대−최소 차이는 %p 단위다. 실제 변동성이 다를 수 있어 편중 자체가 오류를 뜻하지는 않는다.
아무 경보도 없으면 경보 비중/lift는 정의할 수 없어 NaN으로 둔다.

In [12]:
# 목적: 임계값 적용, 전체/기업별/구간별 평가 함수를 정의한다.
# 예상 출력: 함수 정의. 실험 결과는 다음 셀에서 DataFrame으로 만들어진다.
DECISION_COLUMNS = ["date", "code", "name", "market", "stock_return", "expected_return",
                    "price_residual", "price_z", "volume_z", "volume_ratio"]


def classify_scores(frame, experiment):
    result = frame[DECISION_COLUMNS + ["fold", "split"]].copy()
    for key in ["experiment_id", "train_window", "z_window", "price_threshold", "volume_threshold"]:
        result[key] = experiment[key]
    result["price_anomaly"] = result["price_z"].abs().ge(float(experiment["price_threshold"]))
    result["volume_anomaly"] = result["volume_z"].ge(float(experiment["volume_threshold"]))
    result["anomaly"] = result["price_anomaly"] | result["volume_anomaly"]
    result["anomaly_type"] = np.select(
        [result["price_anomaly"] & result["volume_anomaly"], result["price_anomaly"], result["volume_anomaly"]],
        ["가격+거래량", "가격", "거래량"], default="정상")
    return result


def summarize_decisions(decisions, targets):
    company = targets[["code", "name", "market"]].merge(
        decisions.groupby("code", as_index=False).agg(
            n_evaluated=("anomaly", "size"), n_anomalies=("anomaly", "sum"),
            n_price=("price_anomaly", "sum"), n_volume=("volume_anomaly", "sum")), on="code", how="left")
    count_cols = ["n_evaluated", "n_anomalies", "n_price", "n_volume"]
    company[count_cols] = company[count_cols].fillna(0).astype(int)
    total = int(company["n_evaluated"].sum())
    alerts = int(company["n_anomalies"].sum())
    company["detection_rate_pct"] = 100 * company["n_anomalies"] / company["n_evaluated"].replace(0, np.nan)
    company["exposure_share_pct"] = 100 * company["n_evaluated"] / (total or np.nan)
    company["anomaly_share_pct"] = 100 * company["n_anomalies"] / (alerts or np.nan)
    company["concentration_lift"] = company["anomaly_share_pct"] / company["exposure_share_pct"]
    rates = company["detection_rate_pct"].dropna()
    overall = {
        "n_evaluated": total, "n_anomalies": alerts,
        "detection_rate_pct": 100 * alerts / total if total else np.nan,
        "n_price": int(decisions["price_anomaly"].sum()),
        "n_volume": int(decisions["volume_anomaly"].sum()),
        "n_both": int((decisions["price_anomaly"] & decisions["volume_anomaly"]).sum()),
        "company_rate_gap_pp": rates.max() - rates.min() if len(rates) else np.nan,
        "company_rate_std_pp": rates.std(ddof=0) if len(rates) else np.nan,
        "max_company_share_pct": company["anomaly_share_pct"].max(),
        "max_company_lift": company["concentration_lift"].max(),
        "mean_alerts_per_day": alerts / decisions["date"].nunique() if len(decisions) else np.nan,
    }
    return overall, company


def run_experiments(cache, date_map, common_mask, grid, universe_frame):
    decisions_all, summary_rows, company_parts, fold_rows, company_fold_parts = [], [], [], [], []
    target_frame = universe_frame.loc[universe_frame["is_active"]]
    for experiment in grid.to_dict("records"):
        key = (int(experiment["train_window"]), int(experiment["z_window"]))
        scores = cache[key].merge(date_map, on="date", how="inner", validate="many_to_one")
        scores = scores.merge(common_mask, on=["date", "code"], how="left", validate="one_to_one")
        scores = scores.loc[scores["common_valid"].fillna(False) & scores["model_valid"]].copy()
        decisions = classify_scores(scores, experiment)
        if decisions.empty:
            raise ValueError(f"평가 가능한 데이터 없음: {experiment['experiment_id']}")
        overall, company = summarize_decisions(decisions, target_frame)
        overall.update(experiment)
        company = company.assign(**experiment)
        company_parts.append(company)
        fold_rates = []
        for fold, group in decisions.groupby("fold"):
            fold_summary, fold_company = summarize_decisions(group, target_frame)
            fold_rates.append(fold_summary["detection_rate_pct"])
            fold_summary.update(experiment)
            fold_summary["fold"] = fold
            fold_rows.append(fold_summary)
            company_fold_parts.append(fold_company.assign(fold=fold, **experiment))
        overall["fold_rate_std_pp"] = float(np.std(fold_rates, ddof=0))
        summary_rows.append(overall)
        decisions_all.append(decisions)
    return (pd.concat(decisions_all, ignore_index=True), pd.DataFrame(summary_rows),
            pd.concat(company_parts, ignore_index=True), pd.DataFrame(fold_rows),
            pd.concat(company_fold_parts, ignore_index=True))


def show_table(frame, height=480, digits=3):
    """행을 자르지 않고 스크롤 가능한 HTML 표로 출력한다."""
    table = frame.to_html(index=False, escape=True, float_format=lambda x: f"{x:.{digits}f}", na_rep="—")
    display(HTML(f'<div style="max-height:{height}px;overflow:auto;border:1px solid #ddd">{table}</div>'))

In [13]:
# 목적: 81개 설정의 OR 탐지율·기업 편중을 비교한다.
# 예상 출력: validation_summary 81행, 기업별 요약 81×분석기업수 행, fold별 요약 81×3행.
(validation_decisions, validation_summary, validation_company_summary,
 validation_fold_summary, validation_company_fold_summary) = run_experiments(
    feature_cache, validation_date_map, common_valid, experiment_grid, universe)

assert len(validation_summary) == len(experiment_grid)
assert validation_summary["n_evaluated"].nunique() == 1, "설정별 비교 분모가 달라졌습니다."
summary_columns = ["experiment_id", "train_window", "z_window", "price_threshold", "volume_threshold",
                   "n_evaluated", "n_anomalies", "detection_rate_pct", "n_price", "n_volume", "n_both",
                   "company_rate_gap_pp", "max_company_share_pct", "max_company_lift", "fold_rate_std_pp"]
show_table(validation_summary[summary_columns], height=600)

experiment_id,train_window,z_window,price_threshold,volume_threshold,n_evaluated,n_anomalies,detection_rate_pct,n_price,n_volume,n_both,company_rate_gap_pp,max_company_share_pct,max_company_lift,fold_rate_std_pp
R60_Z40_P2.0_V2.0,60,40,2.000,2.000,2262,190,8.400,140,104,54,2.266,18.947,1.134,1.002
R60_Z40_P2.0_V2.5,60,40,2.000,2.500,2262,164,7.250,140,62,38,2.279,20.122,1.204,0.739
R60_Z40_P2.0_V3.0,60,40,2.000,3.000,2262,156,6.897,140,43,27,1.220,18.590,1.112,0.842
R60_Z40_P2.5_V2.0,60,40,2.500,2.000,2262,144,6.366,79,104,39,2.833,20.139,1.205,1.136
R60_Z40_P2.5_V2.5,60,40,2.500,2.500,2262,110,4.863,79,62,31,2.590,20.909,1.251,0.505
R60_Z40_P2.5_V3.0,60,40,2.500,3.000,2262,99,4.377,79,43,23,1.801,19.192,1.148,0.670
R60_Z40_P3.0_V2.0,60,40,3.000,2.000,2262,121,5.349,44,104,27,3.115,21.488,1.286,0.898
R60_Z40_P3.0_V2.5,60,40,3.000,2.500,2262,85,3.758,44,62,21,2.078,20.000,1.197,0.343
R60_Z40_P3.0_V3.0,60,40,3.000,3.000,2262,71,3.139,44,43,16,2.351,22.535,1.349,0.611
R60_Z60_P2.0_V2.0,60,60,2.000,2.000,2262,188,8.311,138,100,50,3.704,20.213,1.210,1.300


In [14]:
# 목적: 기업별 편중을 임계값 조합마다 확인한다.
# 예상 출력: 81개 설정 × 기업별 탐지율 비교표와 상세 분모/경보 비중/lift 표.
company_rate_matrix = validation_company_summary.pivot(
    index=["train_window", "z_window", "price_threshold", "volume_threshold"], columns="name", values="detection_rate_pct")
print("기업별 탐지율(%):")
show_table(company_rate_matrix.reset_index(), height=550)
print("기업별 상세 지표: anomaly_share_pct는 전체 경보 중 해당 기업의 비중입니다.")
show_table(validation_company_summary[["experiment_id", "code", "name", "n_evaluated", "n_anomalies",
                                       "detection_rate_pct", "exposure_share_pct", "anomaly_share_pct", "concentration_lift"]])

기업별 탐지율(%):


name,train_window,z_window,price_threshold,volume_threshold,DB하이텍,SK하이닉스,삼성전자,원익IPS,주성엔지니어링,한미반도체
,60,40,2.000,2.000,9.259,9.524,8.466,7.937,7.937,7.258
,60,40,2.000,2.500,7.407,8.730,7.407,6.614,6.878,6.452
,60,40,2.000,3.000,7.407,7.672,6.614,6.614,6.614,6.452
,60,40,2.500,2.000,7.672,6.614,7.143,5.556,6.349,4.839
,60,40,2.500,2.500,5.291,5.026,6.085,4.233,5.026,3.495
,60,40,2.500,3.000,5.026,3.968,5.026,4.233,4.762,3.226
,60,40,3.000,2.000,6.878,5.291,5.820,4.762,5.556,3.763
,60,40,3.000,2.500,4.497,3.704,4.497,3.175,4.233,2.419
,60,40,3.000,3.000,4.233,2.116,3.439,3.175,3.968,1.882
,60,60,2.000,2.000,10.053,9.788,8.466,6.349,8.730,6.452


기업별 상세 지표: anomaly_share_pct는 전체 경보 중 해당 기업의 비중입니다.


experiment_id,code,name,n_evaluated,n_anomalies,detection_rate_pct,exposure_share_pct,anomaly_share_pct,concentration_lift
R60_Z40_P2.0_V2.0,005930,삼성전자,378,32,8.466,16.711,16.842,1.008
R60_Z40_P2.0_V2.0,000660,SK하이닉스,378,36,9.524,16.711,18.947,1.134
R60_Z40_P2.0_V2.0,000990,DB하이텍,378,35,9.259,16.711,18.421,1.102
R60_Z40_P2.0_V2.0,042700,한미반도체,372,27,7.258,16.446,14.211,0.864
R60_Z40_P2.0_V2.0,240810,원익IPS,378,30,7.937,16.711,15.789,0.945
R60_Z40_P2.0_V2.0,036930,주성엔지니어링,378,30,7.937,16.711,15.789,0.945
R60_Z40_P2.0_V2.5,005930,삼성전자,378,28,7.407,16.711,17.073,1.022
R60_Z40_P2.0_V2.5,000660,SK하이닉스,378,33,8.730,16.711,20.122,1.204
R60_Z40_P2.0_V2.5,000990,DB하이텍,378,28,7.407,16.711,17.073,1.022
R60_Z40_P2.0_V2.5,042700,한미반도체,372,24,6.452,16.446,14.634,0.890


# 6. 가격·거래량 탐지 날짜: 81개 설정 모두 제공
아래 출력은 구간 조합 → 임계값 조합을 펼치면 기업별 가격 날짜 / 거래량 날짜 / OR 날짜를 전부 보여 준다.

날짜 목록을 head()로 자르지 않는다. 같은 날 두 지표가 탐지하면 가격·거래량 목록 양쪽에 나타나고 OR에는 한 번만 나타난다.

validation_signal_dates는 지표별 한 행씩, validation_events는 기업·날짜·설정별 한 행씩 저장한다.
article_lookup_dates는 여러 설정이 잡은 같은 기업·날짜를 하나로 합친 기사 확인용 목록이다.

In [15]:
# 목적: 각 지표의 탐지 날짜와 중복 제거한 사건 목록을 만든다.
# 예상 출력: 메모리 표 4개 및 81개 설정 전체 날짜 카탈로그.
def make_event_tables(decisions):
    events = decisions.loc[decisions["anomaly"]].sort_values(["experiment_id", "date", "code"]).reset_index(drop=True)
    indicator_parts = []
    for flag, label, zcol in [("price_anomaly", "price", "price_z"), ("volume_anomaly", "volume", "volume_z")]:
        part = decisions.loc[decisions[flag], ["experiment_id", "train_window", "z_window", "price_threshold",
                                               "volume_threshold", "date", "code", "name", "fold", "split", zcol]].copy()
        part = part.rename(columns={zcol: "z_score"})
        part["indicator"] = label
        indicator_parts.append(part)
    signal_dates = pd.concat(indicator_parts, ignore_index=True).sort_values(["experiment_id", "date", "code", "indicator"])
    catalog_rows = []
    for keys, group in decisions.groupby(["experiment_id", "train_window", "z_window", "price_threshold", "volume_threshold", "code", "name"], sort=False):
        row = dict(zip(["experiment_id", "train_window", "z_window", "price_threshold", "volume_threshold", "code", "name"], keys))
        for flag, col in [("price_anomaly", "price_dates"), ("volume_anomaly", "volume_dates"), ("anomaly", "or_dates")]:
            row[col] = tuple(group.loc[group[flag], "date"].sort_values().dt.strftime("%Y-%m-%d"))
        catalog_rows.append(row)
    catalog = pd.DataFrame(catalog_rows)
    lookup = events.groupby(["date", "code", "name"], as_index=False).agg(
        configurations_hit=("experiment_id", "nunique"), price_configurations_hit=("price_anomaly", "sum"),
        volume_configurations_hit=("volume_anomaly", "sum"))
    return events, signal_dates.reset_index(drop=True), catalog, lookup


def show_all_date_catalog(catalog):
    sections = ['<p>구간 조합과 임계값 조합을 눌러 날짜 전체를 확인하세요.</p>']
    for (train_window, z_window), window_group in catalog.groupby(["train_window", "z_window"], sort=True):
        combination_count = len(window_group[["price_threshold", "volume_threshold"]].drop_duplicates())
        sections.append(f'<details><summary><b>회귀 {train_window}일 / Z {z_window}일 — {combination_count}개 임계값 조합</b></summary>')
        for (price_t, volume_t), group in window_group.groupby(["price_threshold", "volume_threshold"], sort=True):
            sections.append(f'<details style="margin:8px 16px"><summary>가격 {price_t:.1f} / 거래량 {volume_t:.1f}</summary>')
            sections.append('<table style="width:100%;table-layout:fixed"><thead><tr><th style="width:15%">기업</th><th>가격 탐지 날짜</th><th>거래량 탐지 날짜</th><th>OR 탐지 날짜</th></tr></thead><tbody>')
            for row in group.itertuples(index=False):
                sections.append(f'<tr><td>{html.escape(row.name)}<br>{row.code}</td>')
                for dates in [row.price_dates, row.volume_dates, row.or_dates]:
                    text = '<br>'.join(html.escape(d) for d in dates) if dates else '없음'
                    sections.append(f'<td style="vertical-align:top">{len(dates)}건<br>{text}</td>')
                sections.append('</tr>')
            sections.append('</tbody></table></details>')
        sections.append('</details>')
    display(HTML(''.join(sections)))


validation_events, validation_signal_dates, validation_date_catalog, article_lookup_dates = make_event_tables(validation_decisions)
print(f"설정별 OR 이벤트: {len(validation_events):,}행 / 지표별 탐지: {len(validation_signal_dates):,}행")
print(f"기사 확인용 중복 제거 기업·날짜: {len(article_lookup_dates):,}행")
show_all_date_catalog(validation_date_catalog)
show_table(article_lookup_dates, height=400)

설정별 OR 이벤트: 9,362행 / 지표별 탐지: 11,610행
기사 확인용 중복 제거 기업·날짜: 250행


기업,가격 탐지 날짜,거래량 탐지 날짜,OR 탐지 날짜
SK하이닉스000660,28건2023-06-192023-07-132023-07-272023-07-312023-10-052023-10-262023-11-142024-01-182024-01-252024-02-062024-02-132024-02-222024-02-232024-02-272024-03-062024-03-202024-03-212024-05-312024-06-072024-06-242024-07-042024-07-252024-08-052024-10-282024-10-312024-11-142024-11-182024-12-09,17건2023-07-272023-07-282023-10-052023-10-172023-11-062023-12-142024-02-082024-02-132024-02-222024-02-232024-03-212024-07-042024-07-182024-07-252024-08-022024-08-052024-09-19,36건2023-06-192023-07-132023-07-272023-07-282023-07-312023-10-052023-10-172023-10-262023-11-062023-11-142023-12-142024-01-182024-01-252024-02-062024-02-082024-02-132024-02-222024-02-232024-02-272024-03-062024-03-202024-03-212024-05-312024-06-072024-06-242024-07-042024-07-182024-07-252024-08-022024-08-052024-09-192024-10-282024-10-312024-11-142024-11-182024-12-09
DB하이텍000990,25건2023-07-132023-10-262023-11-102023-11-152023-11-272023-11-292023-11-302023-12-272024-01-022024-03-042024-03-192024-04-012024-05-222024-05-272024-06-132024-06-172024-06-202024-06-212024-09-232024-09-242024-10-312024-11-192024-12-032024-12-092024-12-10,21건2023-10-262023-11-062023-11-272023-11-302023-12-222023-12-282024-01-022024-03-142024-04-012024-04-022024-05-232024-05-272024-06-132024-06-172024-06-182024-06-192024-06-202024-06-212024-09-242024-11-182024-12-11,35건2023-07-132023-10-262023-11-062023-11-102023-11-152023-11-272023-11-292023-11-302023-12-222023-12-272023-12-282024-01-022024-03-042024-03-142024-03-192024-04-012024-04-022024-05-222024-05-232024-05-272024-06-132024-06-172024-06-182024-06-192024-06-202024-06-212024-09-232024-09-242024-10-312024-11-182024-11-192024-12-032024-12-092024-12-102024-12-11
삼성전자005930,21건2023-07-132023-07-262023-07-272023-07-282023-09-012023-11-062024-01-032024-01-092024-01-192024-02-082024-03-202024-04-022024-04-222024-07-152024-08-062024-10-142024-10-242024-10-282024-10-292024-11-152024-11-18,18건2023-07-262023-07-272023-09-012023-09-042023-10-112023-12-142024-01-092024-01-112024-03-202024-03-212024-04-022024-05-272024-07-042024-07-052024-08-052024-08-062024-09-192024-11-13,32건2023-07-132023-07-262023-07-272023-07-282023-09-012023-09-042023-10-112023-11-062023-12-142024-01-032024-01-092024-01-112024-01-192024-02-082024-03-202024-03-212024-04-022024-04-222024-05-272024-07-042024-07-052024-07-152024-08-052024-08-062024-09-192024-10-142024-10-242024-10-282024-10-292024-11-132024-11-152024-11-18
주성엔지니어링036930,22건2023-06-202023-06-292023-07-172023-07-242023-08-012023-08-022023-09-142023-11-162023-12-042024-01-022024-02-272024-02-282024-02-292024-06-052024-06-072024-06-122024-06-282024-07-042024-07-052024-08-052024-10-292024-11-01,18건2023-06-202023-06-292023-07-172023-07-282023-09-142023-11-062023-11-072024-01-252024-02-062024-02-272024-02-282024-06-052024-06-282024-10-292024-10-302024-10-312024-11-012024-11-04,30건2023-06-202023-06-292023-07-172023-07-242023-07-282023-08-012023-08-022023-09-142023-11-062023-11-072023-11-162023-12-042024-01-022024-01-252024-02-062024-02-272024-02-282024-02-292024-06-052024-06-072024-06-122024-06-282024-07-042024-07-052024-08-052024-10-292024-10-302024-10-312024-11-012024-11-04
한미반도체042700,22건2023-07-132023-07-192023-08-182023-11-062023-11-132024-01-162024-01-182024-01-292024-02-082024-03-262024-03-282024-06-032024-06-122024-08-122024-08-142024-08-162024-08-282024-09-202024-10-172024-10-182024-10-312024-12-11,13건2023-07-132023-07-142023-08-182023-11-072023-11-132024-01-162024-02-082024-02-132024-02-262024-03-042024-06-032024-10-172024-10-18,27건2023-07-132023-07-142023-07-192023-08-182023-11-062023-11-072023-11-132024-01-162024-01-182024-01-292024-02-082024-02-132024-02-262024-03-042024-03-262024-03-282024-06-032024-06-122024-08-122024-08-142024-08-162024-08-282024-09-202024-10-172024-10-182024-10-312024-12-11
원익IPS240810,22건2023-07-102023-07-202023-07-242023-07-272023-07-282023-08-012023-08-102023-11-022024-01-222024-02-012024-02-052024-02-132024-02-292024-03-202024-03-292024-06-182024-07-042024-07-1220

date,code,name,configurations_hit,price_configurations_hit,volume_configurations_hit
2023-06-19,000660,SK하이닉스,6,6,0
2023-06-20,036930,주성엔지니어링,54,54,27
2023-06-29,036930,주성엔지니어링,51,33,45
2023-07-10,240810,원익IPS,45,45,0
2023-07-13,000660,SK하이닉스,45,45,0
2023-07-13,000990,DB하이텍,51,51,0
2023-07-13,005930,삼성전자,39,39,0
2023-07-13,042700,한미반도체,81,81,45
2023-07-14,042700,한미반도체,48,21,36
2023-07-17,036930,주성엔지니어링,81,81,72


In [16]:
# 목적: 원하는 학습/Z 구간에서 9개 임계값의 실제 탐지 행을 자세히 확인한다.
# 예상 출력: 조합별 요약과 날짜·기업·가격 Z·거래량 Z·각 지표 탐지 여부. 모든 행을 표시한다.
# 아래 함수의 indicator는 "all", "price", "volume" 중 선택한다. code=None이면 전체 기업이다.
def inspect_window(train_window=120, z_window=60, indicator="all", code=None):
    if indicator not in {"all", "price", "volume"}:
        raise ValueError("indicator는 all/price/volume 중 하나여야 합니다.")
    mask = validation_summary["train_window"].eq(train_window) & validation_summary["z_window"].eq(z_window)
    show_table(validation_summary.loc[mask, summary_columns], height=320)
    flag = {"all": "anomaly", "price": "price_anomaly", "volume": "volume_anomaly"}[indicator]
    data = validation_decisions.loc[validation_decisions["train_window"].eq(train_window) &
                                    validation_decisions["z_window"].eq(z_window)]
    if code is not None:
        data = data.loc[data["code"].eq(str(code).zfill(6))]
    for price_t, volume_t in itertools.product(PRICE_THRESHOLDS, VOLUME_THRESHOLDS):
        rows = data.loc[data["price_threshold"].eq(price_t) & data["volume_threshold"].eq(volume_t) & data[flag]]
        print(f"가격 {price_t:.1f} / 거래량 {volume_t:.1f} : {len(rows)}건 ({indicator})")
        show_table(rows.sort_values(["date", "code"])[["date", "code", "name", "fold", "stock_return", "expected_return",
                                                           "price_residual", "price_z", "volume_z", "volume_ratio",
                                                           "price_anomaly", "volume_anomaly", "anomaly_type"]], height=300)


inspect_window(train_window=120, z_window=60, indicator="all", code=None)

experiment_id,train_window,z_window,price_threshold,volume_threshold,n_evaluated,n_anomalies,detection_rate_pct,n_price,n_volume,n_both,company_rate_gap_pp,max_company_share_pct,max_company_lift,fold_rate_std_pp
R120_Z60_P2.0_V2.0,120,60,2.000,2.000,2262,186,8.223,139,100,53,3.439,19.892,1.190,1.540
R120_Z60_P2.0_V2.5,120,60,2.000,2.500,2262,164,7.250,139,60,35,2.552,19.512,1.168,1.197
R120_Z60_P2.0_V3.0,120,60,2.000,3.000,2262,150,6.631,139,34,23,2.560,20.000,1.197,1.232
R120_Z60_P2.5_V2.0,120,60,2.500,2.000,2262,132,5.836,76,100,44,2.842,20.455,1.224,1.304
R120_Z60_P2.5_V2.5,120,60,2.500,2.500,2262,106,4.686,76,60,30,1.792,19.811,1.186,0.898
R120_Z60_P2.5_V3.0,120,60,2.500,3.000,2262,89,3.935,76,34,21,2.069,21.348,1.278,1.054
R120_Z60_P3.0_V2.0,120,60,3.000,2.000,2262,112,4.951,40,100,28,2.590,20.536,1.229,1.280
R120_Z60_P3.0_V2.5,120,60,3.000,2.500,2262,80,3.537,40,60,20,2.078,21.250,1.272,0.680
R120_Z60_P3.0_V3.0,120,60,3.000,3.000,2262,60,2.653,40,34,14,2.624,25.000,1.496,0.744


가격 2.0 / 거래량 2.0 : 186건 (all)


date,code,name,fold,stock_return,expected_return,price_residual,price_z,volume_z,volume_ratio,price_anomaly,volume_anomaly,anomaly_type
2023-06-20,036930,주성엔지니어링,1,0.076,0.008,0.068,2.053,1.314,1.985,True,False,가격
2023-06-29,036930,주성엔지니어링,1,0.064,0.014,0.049,2.071,2.115,3.506,True,True,가격+거래량
2023-07-10,240810,원익IPS,1,0.003,-0.030,0.033,2.628,-0.552,0.717,True,False,가격
2023-07-13,000660,SK하이닉스,1,-0.011,0.028,-0.039,-2.598,0.142,0.963,True,False,가격
2023-07-13,000990,DB하이텍,1,-0.008,0.056,-0.065,-3.159,-0.944,0.580,True,False,가격
2023-07-13,005930,삼성전자,1,0.000,0.018,-0.018,-2.385,0.507,1.102,True,False,가격
2023-07-13,042700,한미반도체,1,0.300,0.011,0.289,8.595,2.763,4.848,True,True,가격+거래량
2023-07-14,042700,한미반도체,1,0.136,0.018,0.118,2.208,2.627,4.715,True,True,가격+거래량
2023-07-17,036930,주성엔지니어링,1,0.130,0.049,0.081,4.475,4.188,10.535,True,True,가격+거래량
2023-07-19,042700,한미반도체,1,-0.122,-0.018,-0.104,-2.234,1.021,1.559,True,False,가격


가격 2.0 / 거래량 2.5 : 164건 (all)


date,code,name,fold,stock_return,expected_return,price_residual,price_z,volume_z,volume_ratio,price_anomaly,volume_anomaly,anomaly_type
2023-06-20,036930,주성엔지니어링,1,0.076,0.008,0.068,2.053,1.314,1.985,True,False,가격
2023-06-29,036930,주성엔지니어링,1,0.064,0.014,0.049,2.071,2.115,3.506,True,False,가격
2023-07-10,240810,원익IPS,1,0.003,-0.030,0.033,2.628,-0.552,0.717,True,False,가격
2023-07-13,000660,SK하이닉스,1,-0.011,0.028,-0.039,-2.598,0.142,0.963,True,False,가격
2023-07-13,000990,DB하이텍,1,-0.008,0.056,-0.065,-3.159,-0.944,0.580,True,False,가격
2023-07-13,005930,삼성전자,1,0.000,0.018,-0.018,-2.385,0.507,1.102,True,False,가격
2023-07-13,042700,한미반도체,1,0.300,0.011,0.289,8.595,2.763,4.848,True,True,가격+거래량
2023-07-14,042700,한미반도체,1,0.136,0.018,0.118,2.208,2.627,4.715,True,True,가격+거래량
2023-07-17,036930,주성엔지니어링,1,0.130,0.049,0.081,4.475,4.188,10.535,True,True,가격+거래량
2023-07-19,042700,한미반도체,1,-0.122,-0.018,-0.104,-2.234,1.021,1.559,True,False,가격


...

date,code,name,fold,stock_return,expected_return,price_residual,price_z,volume_z,volume_ratio,price_anomaly,volume_anomaly,anomaly_type
2023-06-29,036930,주성엔지니어링,1,0.064,0.014,0.049,2.071,2.115,3.506,False,True,거래량
2023-07-13,000990,DB하이텍,1,-0.008,0.056,-0.065,-3.159,-0.944,0.580,True,False,가격
2023-07-13,042700,한미반도체,1,0.300,0.011,0.289,8.595,2.763,4.848,True,True,가격+거래량
2023-07-14,042700,한미반도체,1,0.136,0.018,0.118,2.208,2.627,4.715,False,True,거래량
2023-07-17,036930,주성엔지니어링,1,0.130,0.049,0.081,4.475,4.188,10.535,True,True,가격+거래량
2023-07-24,036930,주성엔지니어링,1,0.076,-0.000,0.076,3.601,1.611,2.278,True,False,가격
2023-07-26,005930,삼성전자,1,-0.003,-0.023,0.020,2.532,3.301,2.293,False,True,거래량
2023-07-27,000660,SK하이닉스,1,0.097,0.015,0.083,4.519,2.601,2.628,True,True,가격+거래량
2023-07-27,005930,삼성전자,1,0.027,0.012,0.015,1.849,2.254,1.818,False,True,거래량
2023-07-27,036930,주성엔지니어링,1,0.120,0.112,0.008,0.235,2.030,3.270,False,True,거래량


가격 3.0 / 거래량 2.5 : 80건 (all)


date,code,name,fold,stock_return,expected_return,price_residual,price_z,volume_z,volume_ratio,price_anomaly,volume_anomaly,anomaly_type
2023-07-13,000990,DB하이텍,1,-0.008,0.056,-0.065,-3.159,-0.944,0.580,True,False,가격
2023-07-13,042700,한미반도체,1,0.300,0.011,0.289,8.595,2.763,4.848,True,True,가격+거래량
2023-07-14,042700,한미반도체,1,0.136,0.018,0.118,2.208,2.627,4.715,False,True,거래량
2023-07-17,036930,주성엔지니어링,1,0.130,0.049,0.081,4.475,4.188,10.535,True,True,가격+거래량
2023-07-24,036930,주성엔지니어링,1,0.076,-0.000,0.076,3.601,1.611,2.278,True,False,가격
2023-07-26,005930,삼성전자,1,-0.003,-0.023,0.020,2.532,3.301,2.293,False,True,거래량
2023-07-27,000660,SK하이닉스,1,0.097,0.015,0.083,4.519,2.601,2.628,True,True,가격+거래량
2023-07-28,240810,원익IPS,1,0.094,0.037,0.058,4.011,4.870,8.433,True,True,가격+거래량
2023-08-01,036930,주성엔지니어링,1,0.070,-0.015,0.085,3.404,1.454,2.237,True,False,가격
2023-08-02,036930,주성엔지니어링,1,-0.129,-0.048,-0.082,-3.269,1.250,1.918,True,False,가격


가격 3.0 / 거래량 3.0 : 60건 (all)


date,code,name,fold,stock_return,expected_return,price_residual,price_z,volume_z,volume_ratio,price_anomaly,volume_anomaly,anomaly_type
2023-07-13,000990,DB하이텍,1,-0.008,0.056,-0.065,-3.159,-0.944,0.580,True,False,가격
2023-07-13,042700,한미반도체,1,0.300,0.011,0.289,8.595,2.763,4.848,True,False,가격
2023-07-17,036930,주성엔지니어링,1,0.130,0.049,0.081,4.475,4.188,10.535,True,True,가격+거래량
2023-07-24,036930,주성엔지니어링,1,0.076,-0.000,0.076,3.601,1.611,2.278,True,False,가격
2023-07-26,005930,삼성전자,1,-0.003,-0.023,0.020,2.532,3.301,2.293,False,True,거래량
2023-07-27,000660,SK하이닉스,1,0.097,0.015,0.083,4.519,2.601,2.628,True,False,가격
2023-07-28,240810,원익IPS,1,0.094,0.037,0.058,4.011,4.870,8.433,True,True,가격+거래량
2023-08-01,036930,주성엔지니어링,1,0.070,-0.015,0.085,3.404,1.454,2.237,True,False,가격
2023-08-02,036930,주성엔지니어링,1,-0.129,-0.048,-0.082,-3.269,1.250,1.918,True,False,가격
2023-08-10,240810,원익IPS,1,-0.108,-0.025,-0.082,-4.870,2.429,2.970,True,False,가격


In [17]:
# 목적: 원하면 드롭다운으로 학습/Z 구간을 바꾸어 9개 임계값의 날짜를 조회한다.
# 예상 출력: ipywidgets가 있으면 조회 버튼. 없으면 바로 위 inspect_window()를 직접 호출하면 된다.
try:
    import ipywidgets as widgets
    widgets.interact_manual.options(manual_name="9개 조합 조회")(inspect_window,
        train_window=widgets.Dropdown(options=TRAIN_WINDOWS, value=120, description="회귀 구간"),
        z_window=widgets.Dropdown(options=Z_WINDOWS, value=60, description="Z 구간"),
        indicator=widgets.Dropdown(options=[("전체 OR", "all"), ("가격", "price"), ("거래량", "volume")], description="지표"),
        code=widgets.Dropdown(options=[("전체 기업", None)] + [(r.name, r.code) for r in universe.loc[universe["is_active"]].itertuples()], description="기업"))
except ImportError:
    print("선택 UI 없음: inspect_window(60, 40, indicator='price', code='000660')처럼 호출하세요.")

interactive(children=(Dropdown(description='회귀 구간', index=1, options=(60, 120, 180), value=120), Dropdown(desc…

# 7. 선택한 설정으로 최종 테스트
검증 표와 탐지 날짜를 검토한 뒤 한 설정을 선택하고 RUN_FINAL_TEST=True로 바꾼다.
아래 초기 선택값은 기존 기준 모델일 뿐, 자동 선정한 최적값이 아니다.
테스트 결과를 보고 설정을 반복 변경하면 그 기간도 검증에 사용한 셈이므로 새로운 미사용 테스트가 필요하다.
최종 테스트에서는 선택한 설정 자체가 평가 가능한 표본을 사용한다. 검증의 공통 표본과 분모 정의가 다름을 coverage에 명시한다.

In [19]:
# 목적: 테스트를 열기 전에 모델 설정을 확정한다.
# 예상 출력: 선택한 설정의 검증 지표. RUN_FINAL_TEST=False이면 테스트 점수는 계산하지 않는다.
SELECTED_TRAIN_WINDOW = 120
SELECTED_Z_WINDOW = 60
SELECTED_PRICE_THRESHOLD = 2.5
SELECTED_VOLUME_THRESHOLD = 2.5
RUN_FINAL_TEST = True

selection = experiment_grid.loc[
    experiment_grid["train_window"].eq(SELECTED_TRAIN_WINDOW) & experiment_grid["z_window"].eq(SELECTED_Z_WINDOW) &
    experiment_grid["price_threshold"].eq(SELECTED_PRICE_THRESHOLD) & experiment_grid["volume_threshold"].eq(SELECTED_VOLUME_THRESHOLD)]
if len(selection) != 1:
    raise ValueError("81개 실험 중 하나를 선택하세요.")
selected_experiment = selection.iloc[0].to_dict()
display(validation_summary.loc[validation_summary["experiment_id"].eq(selected_experiment["experiment_id"])])

test_features = pd.DataFrame()
test_decisions = pd.DataFrame()
test_summary = pd.DataFrame()
test_company_summary = pd.DataFrame()
test_events = pd.DataFrame()
test_signal_dates = pd.DataFrame()
test_date_catalog = pd.DataFrame()
test_coverage_report = pd.DataFrame()

if RUN_FINAL_TEST:
    final_cache = build_feature_cache(model_panel, [SELECTED_TRAIN_WINDOW], [SELECTED_Z_WINDOW])
    full_selected_features = final_cache[(SELECTED_TRAIN_WINDOW, SELECTED_Z_WINDOW)]
    test_features = full_selected_features.loc[full_selected_features["date"] >= pd.Timestamp(TEST_START)].copy()
    test_coverage_report = test_features.groupby(["code", "name"], as_index=False).agg(
        requested_rows=("date", "size"), evaluated_rows=("model_valid", "sum"))
    test_coverage_report["coverage_pct"] = 100 * test_coverage_report["evaluated_rows"] / test_coverage_report["requested_rows"]
    test_date_map = pd.DataFrame({"date": test_dates, "fold": 0, "split": "test"})
    final_mask = full_selected_features[["date", "code", "model_valid"]].rename(columns={"model_valid": "common_valid"})
    test_decisions, test_summary, test_company_summary, _, _ = run_experiments(
        final_cache, test_date_map, final_mask, selection, universe)
    test_events, test_signal_dates, test_date_catalog, _ = make_event_tables(test_decisions)
    print("최종 테스트 — 선택한 설정의 유효 기업·날짜 기준")
    display(test_coverage_report)
    show_table(test_summary[summary_columns])
    show_table(test_company_summary)
    show_all_date_catalog(test_date_catalog)
else:
    print("테스트 미실행. 검증 결과로 설정을 선택한 후 RUN_FINAL_TEST=True로 변경하세요.")

,n_evaluated,n_anomalies,detection_rate_pct,n_price,n_volume,n_both,company_rate_gap_pp,company_rate_std_pp,max_company_share_pct,max_company_lift,mean_alerts_per_day,experiment_id,train_window,z_window,price_threshold,volume_threshold,fold_rate_std_pp
40,2262,106,4.686118,76,60,30,1.792115,0.608273,19.811321,1.185535,0.280423,R120_Z60_P2.5_V2.5,120,60,2.5,2.5,0.897605


회귀 120일 / Z [60]일 계산 완료
최종 테스트 — 선택한 설정의 유효 기업·날짜 기준


,code,name,requested_rows,evaluated_rows,coverage_pct
0,000660,SK하이닉스,242,242,100.0
1,000990,DB하이텍,242,242,100.0
2,005930,삼성전자,242,242,100.0
3,036930,주성엔지니어링,242,242,100.0
4,042700,한미반도체,242,242,100.0
5,240810,원익IPS,242,242,100.0


experiment_id,train_window,z_window,price_threshold,volume_threshold,n_evaluated,n_anomalies,detection_rate_pct,n_price,n_volume,n_both,company_rate_gap_pp,max_company_share_pct,max_company_lift,fold_rate_std_pp
R120_Z60_P2.5_V2.5,120,60,2.500,2.500,1452,66,4.545,44,42,20,4.545,21.212,1.273,0.000


code,name,market,n_evaluated,n_anomalies,n_price,n_volume,detection_rate_pct,exposure_share_pct,anomaly_share_pct,concentration_lift,experiment_id,train_window,z_window,price_threshold,volume_threshold
005930,삼성전자,KOSPI,242,11,8,3,4.545,16.667,16.667,1.000,R120_Z60_P2.5_V2.5,120,60,2.500,2.500
000660,SK하이닉스,KOSPI,242,3,2,2,1.240,16.667,4.545,0.273,R120_Z60_P2.5_V2.5,120,60,2.500,2.500
000990,DB하이텍,KOSPI,242,13,11,8,5.372,16.667,19.697,1.182,R120_Z60_P2.5_V2.5,120,60,2.500,2.500
042700,한미반도체,KOSPI,242,14,9,10,5.785,16.667,21.212,1.273,R120_Z60_P2.5_V2.5,120,60,2.500,2.500
240810,원익IPS,KOSDAQ,242,14,8,11,5.785,16.667,21.212,1.273,R120_Z60_P2.5_V2.5,120,60,2.500,2.500
036930,주성엔지니어링,KOSDAQ,242,11,6,8,4.545,16.667,16.667,1.000,R120_Z60_P2.5_V2.5,120,60,2.500,2.500


기업,가격 탐지 날짜,거래량 탐지 날짜,OR 탐지 날짜
SK하이닉스000660,2건2025-05-022025-07-17,2건2025-01-312025-07-17,3건2025-01-312025-05-022025-07-17
DB하이텍000990,11건2025-02-072025-02-102025-03-062025-06-022025-06-182025-08-262025-09-102025-09-162025-10-022025-11-102025-11-11,8건2025-02-072025-02-102025-09-102025-09-112025-09-162025-10-102025-11-102025-11-11,13건2025-02-072025-02-102025-03-062025-06-022025-06-182025-08-262025-09-102025-09-112025-09-162025-10-022025-10-102025-11-102025-11-11
삼성전자005930,8건2025-03-172025-05-122025-06-162025-07-032025-07-282025-09-222025-11-042025-12-26,3건2025-07-172025-10-022025-10-31,11건2025-03-172025-05-122025-06-162025-07-032025-07-172025-07-282025-09-222025-10-022025-10-312025-11-042025-12-26
주성엔지니어링036930,6건2025-02-142025-02-192025-03-172025-07-312025-08-282025-12-01,8건2025-02-142025-02-192025-03-172025-03-202025-03-212025-09-182025-09-192025-10-02,11건2025-02-142025-02-192025-03-172025-03-202025-03-212025-07-312025-08-282025-09-182025-09-192025-10-022025-12-01
한미반도체042700,9건2025-01-082025-01-222025-02-102025-04-232025-05-162025-07-312025-09-302025-10-012025-10-10,10건2025-01-072025-01-082025-01-222025-02-192025-04-232025-06-192025-09-302025-10-022025-10-102025-10-13,14건2025-01-072025-01-082025-01-222025-02-102025-02-192025-04-232025-05-162025-06-192025-07-312025-09-302025-10-012025-10-022025-10-102025-10-13
원익IPS240810,8건2025-02-112025-05-092025-06-112025-07-032025-07-282025-08-082025-10-022025-12-18,11건2025-02-112025-02-192025-02-212025-04-112025-06-112025-07-282025-07-292025-08-072025-08-082025-08-132025-10-02,14건2025-02-112025-02-192025-02-212025-04-112025-05-092025-06-112025-07-032025-07-282025-07-292025-08-072025-08-082025-08-132025-10-022025-12-18


# 8. 미래 참조 점검
모든 회귀 학습 종료일과 Z 기준 종료일이 평가일보다 앞서는지 확인한다.
추가로 기준일 이후의 가격/거래량/시장/산업 값을 크게 바꾸어도 기준일까지의 점수가 같아야 한다.

이 검사는 데이터의 시간 사용을 확인한다. 현재 공급자가 과거 수정주가를 재작성했는지까지 검증하는 것은 아니다.

In [20]:
# 목적: 시간 참조 경계와 미래 데이터 변경에 대한 불변성을 검사한다.
# 예상 출력: 모든 검사를 통과하면 PASS. 실제 계산 시점의 미래 누수를 발견하면 AssertionError.
for (train_window, z_window), scores in feature_cache.items():
    valid = scores.loc[scores["model_valid"]]
    assert valid["regression_train_end"].lt(valid["date"]).all()
    assert valid["z_reference_end"].lt(valid["date"]).all()
    assert valid["regression_train_n"].eq(train_window).all()

audit_code = TARGET_CODES[0]
audit_original = development_panel.loc[development_panel["code"].eq(audit_code)].copy()
audit_cutoff = validation_date_map["date"].iloc[len(validation_date_map)//2]
audit_changed = audit_original.copy()
future = audit_changed["date"].gt(audit_cutoff)
audit_changed.loc[future, "stock_return"] += 0.50
audit_changed.loc[future, "market_return"] -= 0.10
audit_changed.loc[future, "sector_return"] += 0.20
audit_changed.loc[future, "volume"] *= 10
audit_changed.loc[future, "log_volume"] = np.log1p(audit_changed.loc[future, "volume"])
audit_new = add_past_zscores(fit_rolling_one_company(audit_changed, 120), 60)
audit_reference = feature_cache[(120, 60)].loc[lambda x: x["code"].eq(audit_code)]
audit_columns = ["alpha", "beta_market", "beta_sector", "expected_return", "price_z", "volume_z"]
left = audit_reference.loc[audit_reference["date"].le(audit_cutoff)].set_index("date")[audit_columns]
right = audit_new.loc[audit_new["date"].le(audit_cutoff)].set_index("date")[audit_columns]
pd.testing.assert_frame_equal(left, right, check_exact=False, atol=1e-12, rtol=1e-12)
leakage_audit = pd.DataFrame([
    {"check": "회귀 종료일 < 평가일", "result": "PASS"},
    {"check": "Z 기준 종료일 < 평가일", "result": "PASS"},
    {"check": "학습 표본 수 = 회귀 구간", "result": "PASS"},
    {"check": "미래 데이터 변조 전후 과거 점수 동일", "result": "PASS"},
])
display(leakage_audit)

,check,result
0,회귀 종료일 < 평가일,PASS
1,Z 기준 종료일 < 평가일,PASS
2,학습 표본 수 = 회귀 구간,PASS
3,미래 데이터 변조 전후 과거 점수 동일,PASS


# 9. 선택: 유가·환율 보조 데이터
FETCH_MACRO=True로 두면 원/달러 환율과 WTI 선물 시세를 FDR로 수집한다.
이번 81개 설정의 핵심 회귀 변수는 시장·산업 요인이다. 보조 지표는 향후 확장 비교에 쓸 수 있도록 독립 표에 보관한다.

유가 선물은 현물과 다르고 계약 교체의 영향이 있을 수 있다. 관측일과 실제 이용 가능 시각도 구분해야 한다.

해외 당일 종가를 같은 날짜의 국내 장 마감 모델에 무조건 합치지 않는다. available_at은 확인 전까지 NaT로 남긴다.

In [ ]:
# 목적: 선택적으로 보조 지표를 수집한다.
# 예상 출력: 지표별 수집 로그와 macro_daily. FETCH_MACRO=False이면 빈 표만 만든다.

"""
macro_daily = pd.DataFrame(columns=["indicator_code", "observation_date", "value", "unit", "source", "available_at", "collected_at"])
macro_download_log = pd.DataFrame(columns=["indicator_code", "status", "error"])
if FETCH_MACRO:
    macro_specs = [("USD/KRW", "KRW per USD"), ("CL=F", "USD per barrel; futures")]
    parts, logs = [], []
    for symbol, unit in macro_specs:
        try:
            raw = read_fdr(symbol, DATA_START, DATA_END)
            values = pd.to_numeric(raw["Close"], errors="coerce")
            parts.append(pd.DataFrame({"indicator_code": symbol, "observation_date": raw.index,
                                       "value": values.values, "unit": unit, "source": "FDR/YAHOO",
                                       "available_at": pd.NaT, "collected_at": pd.Timestamp.now(tz="UTC")}))
            logs.append({"indicator_code": symbol, "status": "ok", "error": ""})
        except Exception as exc:
            logs.append({"indicator_code": symbol, "status": "error", "error": str(exc)})
    if parts:
        macro_daily = pd.concat(parts, ignore_index=True)
    macro_download_log = pd.DataFrame(logs)
display(macro_download_log)
display(macro_daily.head())
"""